# Setup 2 — carga histórica (bootstrap da base)

**Objetivo:** puxar o **máximo de histórico** de cada fonte — alvo: **desde o começo do ano**,
ou o mais perto disso que a fonte ainda entrega (algumas guardam só alguns meses/pregões).

**Rode depois do `setup_1_teste.ipynb`** (que já provou cada fluxo). **Pode dar `Run All`** —
nada aborta; cada bloco mostra `[OK]`/`[FALHA]` e a célula final confere as contagens. Se um
bloco falhar, corrija e **re-rode só ele** (todos idempotentes; NTN-B/cálculo pulam o já feito).

⚠️ **É demorado** (boletim do ano + cálculo dia a dia + **Anbima Data completa por último**).
Dá pra `Run All` e voltar depois. **Rode a partir da pasta `code/`.**

## Config — janela alvo e limites reais de cada fonte

In [ ]:
import sys
from pathlib import Path
from datetime import date, timedelta

if not (Path.cwd() / "scripts").exists():
    raise SystemExit(f"Rode a partir da pasta code/. cwd atual: {Path.cwd()}")
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / "scripts"))
import pipeline_core as pc
from lib.db import get_db

get_db().close()   # garante o schema (o setup_1 ja criou; idempotente)

HOJE    = date.today()
INICIO  = date(HOJE.year, 1, 1)                     # ALVO: comeco do ano
# Limites REAIS de cada fonte (nao adianta pedir alem disso):
INI_IND     = max(INICIO, HOJE - timedelta(days=130))  # deb/NTN-B: fonte guarda ~4 meses
DIAS_DI     = 20    # curva DI B3: ~20 pregoes
DIAS_CRICRA = 5     # CRI/CRA: portal ~5 pregoes

DIAS = [d.isoformat() for d in pc.dias_uteis_entre(INICIO, HOJE)]  # calculo cobre todo o boletim
print("ALVO INICIO (boletim):", INICIO, "| HOJE:", HOJE)
print("INI_IND (deb/NTN-B, limite ~4m):", INI_IND)
print("pregoes no calculo:", len(DIAS))

## Scraping — 1 bloco por fonte
Cada fonte na janela máxima que entrega. **Anbima Data (o mais pesado) por último.**

In [ ]:
# 1. Boletim B3 (negocios) — desde o comeco do ano
pc.boletim(INICIO.isoformat(), HOJE.isoformat())

In [ ]:
# 2. FI Analytics planilha (caracteristicas) — snapshot atual (Playwright+login)
pc.fianalytics()

In [ ]:
# 3. Anbima debentures (indicativas) — ~4 meses (fonte skipa 404 fora da janela)
pc.anbima_deb(INI_IND.isoformat(), HOJE.isoformat())

In [ ]:
# 4. Anbima NTN-B (MtM) — ~4 meses. Duration paralela (4 workers) + skip do ja feito.
pc.ntnb(INI_IND.isoformat(), HOJE.isoformat())

In [ ]:
# 5. Curva DI B3 (MtM) — ultimos ~20 pregoes (limite da fonte)
for d in pc.ultimos_n_dias_uteis(DIAS_DI):
    pc.curva_di(d)

In [ ]:
# 6. Anbima CRI/CRA (indicativas, Playwright) — ultimos ~5 pregoes (limite do portal)
for d in pc.ultimos_n_dias_uteis(DIAS_CRICRA):
    pc.anbima_cricra(d)

In [ ]:
# 7. Outstanding via Bloomberg — SO NO BANCO (no PC pessoal da [FALHA], tudo bem)
pc.outstanding(INICIO.isoformat(), HOJE.isoformat())

In [ ]:
# 8. Anbima Data (caracteristicas + fluxo) — universo COMPLETO. O MAIS PESADO -> por ultimo.
#    Roda antes do calculo (match/spread usam InfoAtivos que este passo preenche).
pc.anbima_data(full=True)

## Cálculo — percorre todos os pregões da janela (`DIAS`)
Idempotente: re-rodar pula o já feito. Ordem: taxa → filtrar → spread Anbima → match → spread over → relatório.

In [ ]:
# 9. Taxa por trade (cascata FI Analytics -> B3) — ja paralelo + cache interno
for X in DIAS:
    pc.calc_taxa(X)

In [ ]:
# 10. Filtrar (VALIDO / FUNDO / BROKER / PF)
for X in DIAS:
    pc.filtrar(X)

In [ ]:
# 11. Spread Anbima das indicativas
for X in DIAS:
    pc.spread_anbima(X)

In [ ]:
# 12. Match de referencia (global, sem data)
pc.match_ref()

In [ ]:
# 13. Spread over dos trades
for X in DIAS:
    pc.spread_over(X)

In [ ]:
# 14. Gerar relatorio (toda a base)
pc.relatorio()

## Conferência — a base foi montada?
Contagens > 0 nas tabelas principais = deu certo. Se algo ficou zerado, veja qual bloco deu `[FALHA]` e re-rode só ele (+ os de cálculo).

In [ ]:
import sqlite3
c = sqlite3.connect('data/trades.db')
for t in ['NegociosBrutos','NegociosProcessados','InfoAtivos','AnbimaIndicativos','MtmAnbima','FluxoAtivos','Outstanding']:
    print(f'  {t:22s}: {c.execute("SELECT COUNT(*) FROM "+t).fetchone()[0]:>9,} linhas')
c.close()